In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv


In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM

try:
    # Take the user's question
    user_question = input("Enter your scientific question: ")

    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.7,
        max_tokens=1000,
    )

    agent = Agent(
        role="science Explainer",
        goal="Explain scientific concepts in an easy-to-understand and practical manner.",
        backstory="You are an highly experienced teacher with 15+ years of explain science topics and pratcial knowledge on science not only the bookish knwoledge",
        llm=llm,
        verbose=True,
        allow_delegation=False,
        max_iter=5,
        memory=False,
    )

    # Define the task with a {task} template placeholder
    task = Task(
        description="Explain this topic or answer this question: {task}",
        expected_output="A simple explanation in simple english",
        agent=agent,
    )

    crew = Crew(
        agents=[agent],
        tasks=[task],
        verbose=True,
    )

    # Pass the dynamic input using the inputs dictionary in kickoff_async
    result = await crew.kickoff_async(inputs={"task": user_question})

    print("\n--- Final Result ---")
    print(result.raw)
except Exception as e:
    print("Error:", e)
